In [ ]:
import pandas as pd
import pandas as pd
import numpy as np
from scipy.stats import hmean

df_xgb = pd.read_csv('XGBoost_metrics.csv')
df_lstm = pd.read_csv('LSTM_metrics.csv')
df_ridge = pd.read_csv('ridge_metrics.csv')
persistence_df = pd.read_csv('persistence_rmse.csv')

In [ ]:
df_xgb

In [ ]:
df_lstm

In [ ]:
df_ridge

In [ ]:
persistence_df

In [ ]:
final_df = pd.concat([df_xgb, df_lstm, df_ridge], ignore_index=True)
final_df

In [ ]:
final_df.drop(columns=['MAE', 'MSE'], axis='columns', inplace=True)
final_df

In [ ]:
rmse_value = persistence_df.loc[0, 'RMSE']
rmse_value


In [ ]:
final_df['skill_score'] = 1 - (final_df['RMSE'] / rmse_value)
final_df

In [ ]:
ranking_df = pd.DataFrame()
ranking_df["Model"] = final_df["Model"]

for col in ["Training Time (s)", "Prediction Time (s)",  "RMSE"]:
    ranking_df[col] = final_df[col].rank(method="min", ascending=True).astype(int)

ranking_df["R-squared"] = final_df["R-squared"].rank(method="min", ascending=False).astype(int) 
ranking_df["skill_score"] = final_df["skill_score"].rank(method="min", ascending=False).astype(int) 

ranking_df

In [ ]:
ranking_df["Row Average"] = ranking_df.iloc[:, 1:].mean(axis=1)

In [ ]:
ranking_df.sort_values(by=['Row Average'])

**Min-Max Scaling**

In [ ]:
def custom_minmax_normalization(df, higher_is_better, cutoff=-0.5):
    df_norm = pd.DataFrame(index=df.index)

    for col in df.columns:
        norm = df[col].copy()
        if col != 'Model':
            x = df[col].copy()

            x = x.clip(lower=cutoff)

            if higher_is_better[col]:
                norm = (x - x.min()) / (x.max() - x.min())
            else:
                norm = (x.max() - x) / (x.max() - x.min())

        df_norm[col] = norm

    return df_norm

def nested_harmonic_score(df, higher_is_better, cutoff=-0.5):
    df_norm = custom_minmax_normalization(df, higher_is_better, cutoff)

    #Performance Score:
    df_norm["Performance_Score"] = df_norm[["R-squared", "RMSE", "skill_score"]].apply(
        lambda row: hmean(row) if all(row > 0) else 0, axis=1
    )

    #Speed Score: H(Train Time, Predict Time)
    df_norm["Speed_Score"] = df_norm[["Training Time (s)", "Prediction Time (s)"]].apply(
        lambda row: hmean(row) if all(row > 0) else 0, axis=1
    )

    #Overall Score:
    df_norm["Overall_Score"] = df_norm[["Performance_Score", "Speed_Score"]].apply(
        lambda row: hmean(row) if all(row > 0) else 0, axis=1
    )

    return df_norm[["Model", "Performance_Score", "Speed_Score", "Overall_Score"]]


higher_is_better = {
    "Training Time (s)": False,
    "Prediction Time (s)": False,
    "R-squared": True,
    "RMSE": False,
    "skill_score": True,
}

result = nested_harmonic_score(final_df, higher_is_better)
result.sort_values("Overall_Score", ascending=False, inplace=True)
result
